
## Create the focal network. 

If this is your first time using the notebook, begin with the "main function", `build_network`, located toward the end of this document. 


In [ ]:
permissions = "fulldata"
#permissions = 'default'
project_name = 'proj_1050_authorship_networks'


In [ ]:
%run Snippets/header_008


In [ ]:
"""
Create author-country tables
"""
from pyspark.sql import functions as func
from collections import Counter

from pyspark.sql.types import ArrayType, StringType

# https://en.wikipedia.org/wiki/Member_state_of_the_European_Union
# UK added for replication
EU_COUNTRIES = [
    "Austria",
    "Belgium",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "Czech Republic",
    "Denmark",
    "Estonia",
    "Finland",
    "France",
    "Germany",
    "Greece",
    "Hungary",
    "Ireland",
    "Italy",
    "Latvia",
    "Lithuania",
    "Luxembourg",
    "Malta",
    "Netherlands",
    "Poland",
    "Portugal",
    "Romania",
    "Slovakia",
    "Slovenia",
    "Spain",
    "Sweden",
    "United Kingdom",
]

# ensure the names match what's in the country table
assert table("static_data.country_20200306").filter(
    func.col("name").isin(EU_COUNTRIES)
).count() == len(EU_COUNTRIES)

def get_author_countries(base_table):
    """
    Parameters
    ----------
        base_table : DataFrame
            The ani table, unexploded

    Returns
    -------
        DataFrame
            DataFrame exploded on auid, with a row for each country/author
            and EU member countries renamed to "EU"
    """
    return (
        # remove publications where ALL authors have links to ALL affiliations
        # these occur rarely when the PDF doesn't specify this linking
        # it has been passed back to the Scopus team to correct these, but they persist in snapshots
        # as a rule of thumb, exclude articles where all possible combinations are included AND the number of these combinations is > 20
        base_table.filter(
            ~(
                (func.size("Au_af") > 20)
                & (func.size("Au_af") == (func.size("Au") * func.size("Af")))
            )
        )
        # links authors to corresponding affiliations
        # first a kind of cartesian join, then clean up correspondences
        .withColumn("auaf", func.explode("Au_af"))
        .withColumn("Au", func.explode("Au"))
        .filter("Au.Authorseq=auaf.Authorseq")
        .withColumn(
            "country", func.expr("Af[auaf.affiliation_seq-1].affiliation_country")
        )
        .withColumn(
            "country",
            func.when(func.col("country").isin(EU_COUNTRIES), "EU").otherwise(
                func.col("country")
            ),
        )
    )

ALLOWED_COUNTRIES = ["Brazil", "EU", "Japan", "United States"]

@func.udf(returnType=ArrayType(StringType()))
def get_countries(country_array):
    """
    Parameters
    ----------
        country_array : ArrayType
            An array of countries

    Returns
    -------
        ArrayType<str>
            List of countries, with those that comprise less that 30% of the author's total
            countries filtered out
    """
    if country_array and len(country_array) > 0:
        counter = Counter(country_array)
        total = len(country_array)
        countries = [
            item[0] for item in counter.most_common() if (item[1] / total) >= 0.3
        ]
        if countries:
            return list(set(countries))

def get_author_country_condensed(base_table, start_year, end_year):
    """
    Parameters
    ----------
        base_table : DataFrame
            The ani table, unexploded
        start_year : int
            The year from which to begin filtering, inclusive
        end_year : int
            The year at which to end filtering, inclusive

    Returns
    -------
        DataFrame
            DataFrame with two columns: auid (int) and countries (Array<str>).
            EU countries are condensed and those not meeting the 30% threshold are filtered out.
    """
    df_countries = get_author_countries(base_table)
    return (
        df_countries.filter(
            (func.col("Year") >= start_year) & (func.col("Year") <= end_year)
        )
        .select("Au.auid", "country")
        .groupBy("auid")
        .agg(func.collect_list(func.col("country")).alias("countries"))
        .withColumn("countries", get_countries(func.col("countries")))
        .filter(func.col("countries").isNotNull())
    )


In [ ]:
from collections import Counter
from pyspark.sql.types import ArrayType, StringType

"""
Create author-subject tables.
Note that each subject area must have at least 30% of total to be included (an author can have multiple subjects)
"""

ALLOWED_SUBJECTS = ["BIOC", "MEDI", "ENGI", "BUSI_ECON"]

@func.udf(returnType=ArrayType(StringType()))
def get_subject_areas(subjarea_array):
    """
    Parameters
    ----------
        subjarea_array : ArrayType<str>
            An array of subject areas

    Returns
    -------
        ArrayType<str>
            List of subject areas, with Business and Econ combined and those that comprise less that 30% of the author's total
            subjects filtered out
    """
    # combine business and econ
    condensed = ["BUSI_ECON" if s in ["ECON", "BUSI"] else s for s in subjarea_array]
    counter = Counter(condensed)
    total = len(condensed)
    subs = [item[0] for item in counter.most_common() if (item[1] / total) >= 0.3]
    if subs:
        return subs

def get_author_subject(ani_table):
    """
    Parameters
    ----------
        ani_table : DataFrame
            The ani table, exploded on auid
    Returns
    -------
        DataFrame
            DataFrame with two columns: auid (int) and subject_areas (Array<str>).
            Business and Econ are combined and those not meeting the 30% threshold are filtered out.
    """
    return (
        ani_table.groupBy("auid")
        .agg(func.collect_list("subjareas").alias("subjareas_all"))
        .withColumn("subjareas_all", func.flatten(func.col("subjareas_all")))
        .withColumn("subject_areas", get_subject_areas(func.col("subjareas_all")))
        .filter(func.col("subject_areas").isNotNull())
        .select("auid", "subject_areas")
        .distinct()
    )


In [ ]:
"""
Create focal and active groups based on following criteria:

MEF:
To replicate the existing dataset:
Criteria for actively publishing authors:
Published twice in 2009-2013 OR
Published first ever publication in 2009-2013

"""


def annotate_base_pubs(df_ani, start_year, end_year, focal_year):
    """
    Parameters
    ----------
        df_ani : DataFrame
            The ani table, unexploded and unfiltered by year
        start_year : int
            The year from which to begin filtering, inclusive
        end_year : int
            The year at which to end filtering, inclusive
        focal_year : int
            The year in which an author must publish in order to be classified as focal

    Returns
    -------
        DataFrame
            DataFrame that's been exploded on auid and filtered by year, with one row for each auid-eid-focal_country-focal_subject combination.
            The columns are:
                auid (int) : the author's id
                countries (Array<str>) : the author's countries of publication during the period (filtered by 30% rule)
                country_focal : the country for the publication if the publication qualified as focal (one per row)
                subject_areas : the author's subject areas during the period (filtered by 30% rule)
                subject_focal : the publication subject area if the publication qualified as focal (one per row)
                gender : the author's gender (includes nulls and unknowns)
                pub_count : the count of the author's publications during the period
                first_pub_year : the author's first year of publication
                Year : the year of publication
                eid : the id of the publication
    """

    df_ani_exploded = df_ani.withColumn("auid", func.explode("au.auid")).filter(
        func.col("auid").isNotNull()
    )

    df_ani_pub_year_filtered = df_ani_exploded.filter(
        (func.col("Year") >= start_year) & (func.col("Year") <= end_year)
    )

    df_pub_counts = (
        df_ani_pub_year_filtered.groupby("auid")
        .agg(func.countDistinct(func.col("eid")).alias("pub_count"))
        .distinct()
    )

    df_gender = spark.read.format("parquet").load(
        f"{basePath}{tablename_gender_inference}"
    )

    # table with first year the author published, keyed by auid
    df_first_pub_year = (
        df_ani_exploded.groupBy(func.col("auid"))
        .agg(func.min(func.col("Year")).alias("first_pub_year"))
        .select("auid", "first_pub_year")
        .distinct()
    )

    df_countries = get_author_countries(df_ani)

    # get publications that would qualify an author as focal
    # note that we might have multiple entries per auid-eid combination
    df_focal_pubs = (
        df_countries.selectExpr("subjareas", "eid", "Au.auid", "country", "Year")
        .filter(func.col("Year") == focal_year)
        .withColumn("subject_focal", func.explode(func.col("subjareas")))
        .withColumn(
            "subject_focal",
            func.when(
                func.col("subject_focal").isin(["BUSI", "ECON"]), "BUSI_ECON"
            ).otherwise(func.col("subject_focal")),
        )
        .filter(func.col("subject_focal").isin(ALLOWED_SUBJECTS))
        .filter(func.col("country").isin(ALLOWED_COUNTRIES))
        .selectExpr("subject_focal", "country as country_focal", "auid", "eid")
        .distinct()
    )

    df_author_country = get_author_country_condensed(df_ani, start_year, end_year)

    df_author_subject = get_author_subject(df_ani_pub_year_filtered)

    return (
        df_ani_pub_year_filtered.alias("a")
        .join(
            df_focal_pubs.alias("fp"),
            (
                (func.col("a.auid") == func.col("fp.auid"))
                & (func.col("a.eid") == func.col("fp.eid"))
            ),
            "leftouter",
        )
        .join(
            df_first_pub_year.alias("p"),
            func.col("a.auid") == func.col("p.auid"),
            "leftouter",
        )
        .join(
            df_pub_counts.alias("co"),
            (func.col("a.auid") == func.col("co.auid")),
            "leftouter",
        )
        # join gender
        .join(
            df_gender.alias("g"), func.col("a.auid") == func.col("g.auid"), "leftouter"
        )
        # join countries
        .join(
            df_author_country.alias("c"),
            func.col("a.auid") == func.col("c.auid"),
            "leftouter",
        )
        # join subject areas
        .join(
            df_author_subject.alias("s"),
            func.col("a.auid") == func.col("s.auid"),
            "leftouter",
        )
        .selectExpr(
            "a.auid",
            "c.countries",
            "country_focal",
            "s.subject_areas",
            "subject_focal",
            "Inferred_Probable_Gender as gender",
            "pub_count",
            "first_pub_year",
            "Year",
            "a.eid",
        )
        .distinct()
    )


def annotate_base_pubs_with_is_focal(annotated_base_pubs, start_year, end_year):
    """
    Parameters
    ----------
        annotated_base_pubs : DataFrame
            Table with active author metadata, typically returned from `annotate_base_pubs`
        start_year : int
            The year from which to begin filtering, inclusive
        end_year : int
            The year at which to end filtering, inclusive

    Returns
    -------
        DataFrame
            DataFrame with input columns plus the additional `is_focal` (boolean) and `is_active` (boolean) fields
    """
    active_set = (
        annotated_base_pubs.alias("a")
        .filter(
            (
                (func.col("first_pub_year") >= start_year)
                & (func.col("first_pub_year") <= end_year)
            )
            | (func.col("pub_count") >= 2)
        )
        .withColumn("is_active", func.lit(True))
        .distinct()
    )

    focal_set = (
        active_set.filter(func.col("first_pub_year") >= 1989)
        .filter(func.col("country_focal").isNotNull())
        .filter(func.col("subject_focal").isNotNull())
        .filter((func.col("gender") != "unknown") & func.col("gender").isNotNull())
        .selectExpr("a.auid")
        .withColumn("is_focal", func.lit(True))
        .distinct()
    )

    return (
        annotated_base_pubs.alias("b")
        .join(
            active_set.selectExpr("auid", "is_active", "eid").distinct().alias("a"),
            (
                (func.col("b.auid") == func.col("a.auid"))
                & (func.col("b.eid") == func.col("a.eid"))
            ),
            "leftouter",
        )
        .withColumn(
            "is_active", func.when(func.col("is_active") == True, True).otherwise(False)
        )
        # don't join on eid b/c one focal pub is all you need for all your pubs to get in the focal set
        .join(
            focal_set.alias("f"),
            (func.col("b.auid") == func.col("f.auid")),
            "leftouter",
        )
        .withColumn(
            "is_focal", func.when(func.col("is_focal") == True, True).otherwise(False)
        )
        .selectExpr("b.*", "is_active", "is_focal")
        .distinct()
    )


In [ ]:
# get table representation of focal authors
def get_focal_cohort_breakdown(focal_rows, low_bound, upper_bound):
    """
    Break the focal authors into 5-year cohorts based on year of first publication and return counts by subject and country

    Parameters
    ----------
        focal_rows : DataFrame
            Focal authors, typically returned from `annotate_base_pubs_with_is_focal` and filtered to only those authors where `is_focal` is true
        low_bound : int
            The first year in the year range
        upper_bound : int
            The last year in the year range

    Returns
    -------
        JSON
            JSON representation of publications grouped by cohort-country-subject
    """

    # e.g., 1989, 2013
    bands = range(low_bound, upper_bound + 5, 5)
    cohorts = [
        str(y) + "-" + str(bands[i + 1] - 1) for i, y in enumerate(bands) if i < len(bands) - 1
    ]

    focal_authors_with_cohort_by_focal_country = (
        focal_rows.withColumn(
            "cohort",
            func.when(
                (func.col("first_pub_year") >= bands[0])
                & (func.col("first_pub_year") <= bands[1] - 1),
                cohorts[0],
            )
            .when(
                (func.col("first_pub_year") >= bands[1])
                & (func.col("first_pub_year") <= bands[2] - 1),
                cohorts[1],
            )
            .when(
                (func.col("first_pub_year") >= bands[2])
                & (func.col("first_pub_year") <= bands[3] - 1),
                cohorts[2],
            )
            .when(
                (func.col("first_pub_year") >= bands[3])
                & (func.col("first_pub_year") <= bands[4] - 1),
                cohorts[3],
            )
            .otherwise(cohorts[4]),
        )
        .select("country_focal", "gender", "cohort", "subject_focal", "auid")
        .distinct()
    )

    grouped = focal_authors_with_cohort_by_focal_country.groupBy(
        ["country_focal", "gender", "cohort", "subject_focal"]
    ).agg(func.count(func.col("auid")).alias("count"))

    return grouped.toPandas().to_json(orient="records")


In [ ]:
def get_dataset_summary_stats(dataset, start_year, end_year):
    """
    Calculate simple summary information about the authors in the network

    Parameters
    ----------
        dataset : DataFrame
            Authors with publications, typically returned from `annotate_base_pubs`
        start_year : int
            The year from which to begin filtering, inclusive
        end_year : int
            The year at which to end filtering, inclusive

    Returns
    -------
        Dict[str, str | float]
            Dictionary keyed by the name of the measure
    """
    result = {}

    active_rows = dataset.filter(func.col("is_active") == True).distinct()

    active_rows.cache()

    focal_rows = dataset.filter(func.col("is_focal") == True).distinct()

    focal_rows.cache()

    result["active_row_count"] = active_rows.count()

    result["focal_row_count"] = focal_rows.count()

    result["focal_author_row_count"] = (
        dataset.selectExpr("auid", "is_focal", "country_focal", "subject_focal")
        .filter(func.col("is_focal") == True)
        .distinct()
        .count()
    )

    base_authors = dataset.selectExpr(
        "auid",
        "is_focal",
        "is_active",
    ).distinct()

    base_authors.cache()

    result["active_author_count"] = base_authors.filter("is_active == True").count()
    result["focal_author_count"] = base_authors.filter("is_focal == True").count()

    gendered_totals = (
        active_rows.groupBy(func.col("gender"))
        .agg(func.countDistinct(func.col("auid")).alias("count"))
        .toPandas()
    )

    gendered_focal_totals = (
        focal_rows.groupBy(func.col("gender"))
        .agg(func.countDistinct(func.col("auid")).alias("count"))
        .toPandas()
    )

    result["pct_male_all"] = gendered_totals.loc[
        gendered_totals["gender"] == "male", "count"
    ].values[0] / sum(gendered_totals["count"])

    result["pct_male_focal"] = gendered_focal_totals.loc[
        gendered_focal_totals["gender"] == "male", "count"
    ].values[0] / sum(gendered_focal_totals["count"])

    result["focal_cohort_breakdown"] = get_focal_cohort_breakdown(
        focal_rows, end_year - 24, end_year
    )

    return result


In [ ]:
def _get_focal_collaborators(focal_set, active_set):
    """
    Get auids of all authors who collaborated with the focal authors

    Parameters
    ----------
        focal_set : DataFrame
            Filtered authors with publications and active/focal labels, typically returned from `annotate_base_pubs_with_is_focal`, where `is_focal` is True
        active_set : DataFrame
            Filtered authors with publications and active/focal labels, typically returned from `annotate_base_pubs_with_is_focal`, where `is_active` is True

    Returns
    -------
        DataFrame
            A one-column dataframe with unique author ids
    """

    return (
        focal_set.alias("f")
        .join(active_set.alias("a"), func.col("f.eid") == func.col("a.eid"))
        .filter(func.col("f.auid") != func.col("a.auid"))
        .selectExpr("a.auid")
        .distinct()
    )


def _get_active_collaborators(focal_collab_set, active_set):
    """
    Parameters
    ----------
        focal_collab_set : DataFrame
            DataFrame filtered to include only rows (`eid` and `auid` required) of authors who collaborated with focal authors
        active_set : DataFrame
            Filtered authors with publications and active/focal labels, typically returned from `annotate_base_pubs_with_is_focal`, where `is_active` is True

    Returns
    -------
        DataFrame
            The network as a DataFrame of edges with three columns:
                src (int) : the auid of the author
                dst (int) : the auid of the collaborator
                relationship (int) : the count of collaborations between the two
            Note that this is an undirected network and src and dst are assigned arbitrarily
    """
    return (
        focal_collab_set.alias("f")
        .join(active_set.alias("a"), func.col("f.eid") == func.col("a.eid"))
        .filter(func.col("f.auid") != func.col("a.auid"))
        .withColumn(
            "_src",
            func.when(
                func.col("f.auid") > func.col("a.auid"), func.col("f.auid")
            ).otherwise(func.col("a.auid")),
        )
        .withColumn(
            "_dst",
            func.when(
                func.col("f.auid") < func.col("a.auid"), func.col("f.auid")
            ).otherwise(func.col("a.auid")),
        )
        .selectExpr("_src as src", "_dst as dst", "a.eid")
        .distinct()
        .groupBy("src", "dst")
        .agg(func.count("eid").alias("relationship"))
        .orderBy(func.col("relationship"))
    )


def get_network(active_labeled):
    """
    Build the network of focal authors and their collaborators and collaborators' collaborators.
    We do this by joining the focal collaborators with their own collaborators on publication ID.
    This should work b/c the focal authors' collaborators' collaborators include the focal authors themselves.

    Parameters
    ----------
        dataset : DataFrame
            Filtered authors with publications and active/focal labels, typically returned from `annotate_base_pubs_with_is_focal`

    Returns
    -------
        DataFrame
            The network as a DataFrame of edges with three columns:
                src (int) : the auid of the author
                dst (int) : the auid of the collaborator
                relationship (int) : the count of collaborations between the two
            Note that this is an undirected network and src and dst are assigned arbitrarily
    """
    focals = active_labeled.filter(func.col("is_focal") == True)
    focal_collabs = _get_focal_collaborators(focals, active_labeled)
    first_order_collabs = (focal_collabs
        .alias("d")
        .join(active_labeled.alias("a"), func.col("d.auid") == func.col("a.auid"))
        .selectExpr("a.*")
    )
    return _get_active_collaborators(first_order_collabs, active_labeled).distinct()


In [ ]:
from pyspark.sql.window import Window

windowSpecAgg = Window.partitionBy("id")

def get_vertices(rows):
    """
    Return one row for each author in the network by dropping columns that distinguish them (`Year`, `eid`, `country_focal`, `subject focal`)
    and collapsing focal subjects and countries into a list.

    Parameters
    ----------
        rows : DataFrame
            Filtered authors with publications and active/focal labels, typically returned from `annotate_base_pubs_with_is_focal`

    Returns
    -------
        DataFrame
    """
    return (
        rows.withColumnRenamed("auid", "id")
        .withColumn(
            "countries_focal",
            func.collect_set(func.col("country_focal")).over(windowSpecAgg),
        )
        .withColumn(
            "subjects_focal",
            func.collect_set(func.col("subject_focal")).over(windowSpecAgg),
        )
        .drop("Year", "eid", "country_focal", "subject_focal")
        .distinct()
    )


In [ ]:
def get_sample(df_ani, size):
    """
    Sample the main publication table

    There are (at least) two related issues with "naive" sampling
    1. The network does not remain intact because authors' publications are arbitrarily removed
    2. Focal and active groupings are distorted b/c qualifying publications are lost

    We can mitigate the above by ensuring that we preserve all of a given author's publications within the sample.
    In other words, we're really sampling authors here, not publications.

    Parameters
    ----------
      df_ani : DataFrame
        ani table, unexploded
      size : float
        Sample size (of authors), a decimal between 0 and 1

    Returns
    -------
      DataFrame

    """
    exploded = df_ani.withColumn("auid", func.explode("au.auid"))
    sample = exploded.selectExpr("auid").distinct().sample(size)
    eids = (
        sample.alias("s")
        .join(exploded.alias("e"), func.col("s.auid") == func.col("e.auid"))
        .selectExpr("e.eid")
        .distinct()
    )
    return (
        df_ani.alias("a")
        .join(eids.alias("e"), func.col("a.eid") == func.col("e.eid"))
        .selectExpr("a.*")
    )


In [ ]:
def build_network(df_ani, start_year, end_year, focal_year, save=False, save_suffix=""):
    """
    Build the focal network --- this is the "main function" in this notebook.

    Parameters
    ----------
        df_ani : DataFrame
            ani table, unexploded
        start_year : int
            The year from which to begin filtering, inclusive
        end_year : int
            The year at which to end filtering, inclusive
        focal_year : int
            The year in which an author must publish in order to be classified as focal
        save : bool
            Whether to save the results to object storage
        save_suffix: str
            A suffix to append to the auto-generated filename
            Example: "-sample-.5pct"

    Returns
    -------
      DataFrame

    """
    print("creating active and focal groups....")
    annotated_base = annotate_base_pubs(df_ani, start_year, end_year, focal_year)

    annotated_base_with_focal = annotate_base_pubs_with_is_focal(
        annotated_base, start_year, end_year
    )

    annotated_base_with_focal.cache()

    print("calculating summary stats....")

    summary_stats = get_dataset_summary_stats(
        annotated_base_with_focal, start_year, end_year
    )

    active_rows = annotated_base_with_focal.filter(func.col("is_active") == True)

    print("building network....")

    edges = get_network(active_rows)

    vertices = get_vertices(active_rows)

    if save:

        print("saving....")

        suffix = str(start_year)[2:] + str(end_year)[2:] + save_suffix

        dbutils.fs.rm(
            f"s3a://elsevier-fcads-icsr-userdata-ro/proj_1050_authorship_networks/edges-{suffix}.parquet",
            recurse=True,
        )
        edges.write.parquet(
            f"s3a://elsevier-fcads-icsr-userdata-ro/proj_1050_authorship_networks/edges-{suffix}.parquet"
        )
        dbutils.fs.rm(
            f"s3a://elsevier-fcads-icsr-userdata-ro/proj_1050_authorship_networks/vertices-{suffix}.parquet",
            recurse=True,
        )
        vertices.write.parquet(
            f"s3a://elsevier-fcads-icsr-userdata-ro/proj_1050_authorship_networks/vertices-{suffix}.parquet"
        )

    return {"edges": edges, "vertices": vertices, "stats": summary_stats}


In [ ]:
"""
Run the code that builds the network
"""

# load the publication table and filter out unwanted publication types
df_ani = (
    spark.read.format("parquet")
    .load(basePath + tablename_ani)
    .filter(func.col("publication_type").isin(["ar", "re", "cp"]))
)

# sample the main table, if needed
sample = get_sample(df_ani, 0.005)

# build the network
#net = build_network(sample, 2009, 2013, 2013, True, "-fullset-.5pct-sample")
net = build_network(df_ani, 2009, 2013, 2013)

# print stats
print(net["stats"])
# default: 'active_row_count': 931,121, 'focal_row_count': 172,348, 'distinct_focal_row_count': 172,348, 'active_author_count': 492,867, 'focal_author_count': 58,996, 'pct_male_all': 0.54, 'pct_male_focal': 0.64
# full: 'active_row_count': 47,107,048, 'focal_row_count': 12,021,596, 'distinct_focal_row_count': 12,021,596, 'active_author_count': 9,234,491, 'focal_author_count': 1,261,812, 'pct_male_all': 0.47, 'pct_male_focal': 0.61

print(net["edges"].count())  # ?? in default, 120,799,242 in full
print(net["vertices"].count())  # 493,069 in default, 9,234,491 in full
# 5 minutes in default,  20 mins full


In [ ]:
"""
Get publication counts by year for focal set in wide format
"""

base = annotate_base_pubs(df_ani, 2009, 2013, 2013)
annotated = annotate_base_pubs_with_is_focal(base, 2009, 2013)
focal_auids = annotated.filter(func.col("is_focal") == True).selectExpr("auid").distinct()
focal_pubs = (
    df_ani.withColumn("auid", func.explode("au.auid"))
    .alias("a")
    .join(focal_auids.alias("f"), func.col("f.auid") == func.col("a.auid"))
)

focal_pubs_grouped = (
    focal_pubs.selectExpr("a.Year", "a.auid", "a.eid")
    .distinct()
    .groupBy(func.col("Year"), func.col("auid"))
    .agg(func.count(func.col("eid")))
)

display(focal_pubs_grouped)


In [ ]:
focal_pubs_by_year_wide = (
    focal_pubs.selectExpr("a.Year", "a.auid", "a.eid")
    .groupBy(func.col("Year"), func.col("auid"))
    .agg(func.count(func.col("eid")))
    .groupBy("auid")
    .pivot("Year")
    .sum("count(eid)")
)

display(focal_pubs_by_year_wide)
